# Policy Gradients & On-Policy Dynamics

Now that we know how returns ($G_t$) and rewards work, we need the core mathematical mechanism: How do we actually update the neural network weights $\theta$ using these scalar rewards?

1. The Objective Function: What Are We Maximizing?In Supervised Learning, we minimize cross-entropy loss against fixed targets.In Reinforcement Learning, we maximize the Expected Return under the policy $\pi_\theta$:$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$Where:$\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ is a full rollout trajectory (the generated token sequence).$P(\tau \mid \theta) = P(s_0) \prod_{t=0}^T \pi_\theta(a_t \mid s_t)$ is the probability that policy $\pi_\theta$ generates this exact sequence.$R(\tau) = \sum_{t=0}^T r_t$ is the total return of the trajectory.2. The Policy Gradient Theorem (The Core Derivative)To perform gradient ascent, we need the gradient $\nabla_\theta J(\theta)$:$$\nabla_\theta J(\theta) = \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \nabla_\theta \int P(\tau \mid \theta) R(\tau) \, d\tau$$The Problem:We cannot directly push the gradient $\nabla_\theta$ inside the integral because the probability distribution itself ($P(\tau \mid \theta)$) depends on $\theta$. Moreover, the environment transition/reward function $R(\tau)$ is a black box — it is non-differentiable (you can't take the derivative of a Python unit test compiler).The Log-Derivative Trick:Recall from calculus that $\nabla f(x) = f(x) \nabla \log f(x)$. Applying this:$$\nabla_\theta P(\tau \mid \theta) = P(\tau \mid \theta) \nabla_\theta \log P(\tau \mid \theta)$$Substitute this back into the integral:$$\nabla_\theta J(\theta) = \int P(\tau \mid \theta) \nabla_\theta \log P(\tau \mid \theta) R(\tau) \, d\tau = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log P(\tau \mid \theta) R(\tau) \right]$$Expanding the trajectory into individual tokens ($a_t$):$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau) \right]$$3. The Intuition Behind REINFORCELook at the loss term: $\nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)$.Case A: Successful generation (R = +1.0)
Loss gradient pushes up log \pi_\theta(a_t | s_t)
--> Increase the probability of ALL tokens generated in this sequence.

Case B: Failed generation (R = 0.0 or -1.0)
Loss gradient pushes down log \pi_\theta(a_t | s_t)
--> Decrease the probability of ALL tokens generated in this sequence.
The log-probability gradient $\nabla_\theta \log \pi_\theta(a_t \mid s_t)$ gives the direction in parameter space to make that token more likely. The scalar reward $R(\tau)$ acts as the magnitude and sign of the step.4. The Fatal Flaw of Pure REINFORCE: Variance & BaselinesSuppose every reward is positive: $R \in [10, 20]$.A mediocre response gets $R = 10$.An exceptional response gets $R = 20$.Under pure REINFORCE, both responses get their probabilities increased because $R > 0$. The model will eventually learn because the better response is pushed up twice as aggressively, but the gradient variance is enormous, leading to extremely unstable training.The Fix: Advantage and BaselinesInstead of multiplying by the raw return $R(\tau)$, we subtract a baseline $b(s_t)$:$$\nabla_\theta J(\theta) = \mathbb{E} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot A(s_t, a_t) \right]$$Where the Advantage is:$$A(s_t, a_t) = R(\tau) - b(s_t)$$If an action produces an outcome better than average ($A > 0$), increase its probability.If an action produces an outcome worse than average ($A < 0$), decrease its probability.Subtracting a baseline that does not depend on the action $a_t$ reduces variance without introducing bias into the gradient expectation.5. On-Policy vs. Off-Policy DynamicsOn-Policy: The data used to compute the gradient $\nabla_\theta J(\theta)$ must be generated by the exact current parameters $\theta$. Once you update $\theta \to \theta_{\text{new}}$, the old generated data is invalid and must be discarded.Why this matters for LLMs:Generating text (rollouts) requires running autoregressive forward passes across multiple GPUs.In on-policy training, rollout generation is the primary compute bottleneck: you generate a batch of samples $\to$ compute gradient $\to$ update weights $\to$ throw away samples $\to$ generate new samples with the updated weights.